# InferScale-X · 02 Model customisation (SFT · DPO · GRPO · RLVR) on one A100
Use a **fresh runtime** (TRL's torch pin differs from vLLM's). LoRA on Qwen2.5-0.5B-Instruct; verifiable arithmetic data with lineage and a leakage check. ~1.5–3 h total.

## 0 · Get the code
Upload `InferScale-X.zip` to your Drive root (or to this session), then run:

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
if not os.path.exists('/content/InferScale-X'):
    !cp /content/drive/MyDrive/InferScale-X.zip /content/ 2>/dev/null || echo 'upload InferScale-X.zip to /content'
    !cd /content && unzip -q -o InferScale-X.zip
%cd /content/InferScale-X
!nvidia-smi

In [ ]:
!bash scripts/colab_setup.sh training

## Data curation → `training/data/lineage.json`

In [ ]:
!python training/data-curation/build_datasets.py && cat training/data/lineage.json

## SFT (~15 min)

In [ ]:
!python training/sft/train_sft.py --max-steps 300

## DPO from base, and SFT→DPO (~20 min each)

In [ ]:
%cd /content/InferScale-X
import glob, os
last = lambda m: sorted(d for d in glob.glob('training/runs/*') if os.path.basename(d).split('-', 2)[-1] == m)[-1]
sft = last('sft'); print(sft)
# DPO + SFT term on chosen (RPO). --sft-weight 0 reproduces plain DPO (collapsed in run 1: see docs/negative-results.md)
!python training/dpo/train_dpo.py --max-steps 200
!python training/dpo/train_dpo.py --max-steps 200 --init-adapter {sft}/adapter


## GRPO with a heuristic (format) reward vs RLVR with a verifiable reward — identical hyper-parameters (~30 min each)

In [ ]:
!python training/grpo/train_grpo.py --reward heuristic --max-steps 150
!python training/rlvr/train_rlvr.py --max-steps 150

## Held-out evaluation: accuracy, format compliance, output length (= serving cost)

In [ ]:
%cd /content/InferScale-X
import glob, os
# exact method match: '*-sft' would also match '<ts>-dpo-from-sft'
last = lambda m: sorted(d for d in glob.glob('training/runs/*') if os.path.basename(d).split('-', 2)[-1] == m)[-1]
have = {m for m in ('sft', 'dpo', 'dpo-rpo', 'dpo-from-sft', 'dpo-from-sft-rpo', 'grpo', 'rlvr')
        if any(os.path.basename(d).split('-', 2)[-1] == m for d in glob.glob('training/runs/*'))}
print(sorted(have))
ads = ','.join(['base'] + [last(m) + '/adapter' for m in ('sft', 'dpo', 'dpo-rpo', 'grpo', 'rlvr') if m in have])
!python training/eval_models.py --adapters {ads}
for m in ('dpo-from-sft', 'dpo-from-sft-rpo'):
    if m in have:
        c, s, o = last(m), last('sft'), 'training/runs/eval_report_' + m.replace('-', '_') + '.json'
        !python training/eval_models.py --adapters {c}/adapter --merge-first {s}/adapter --out {o}


In [ ]:
import json, glob
for f in sorted(glob.glob('training/runs/eval_report*.json')):
    for k, v in json.load(open(f))['results'].items():
        name = k if k == 'base' else k.split('/')[-2].split('-', 2)[-1] + (' (sft merged)' if 'from-sft' in k else '')
        print(f"{name:28s} acc={v['accuracy']:.3f} fmt={v['format_rate']:.3f} out_tok={v['mean_output_tokens']:.1f}")


In [ ]:
!bash scripts/save_results.sh /content/drive/MyDrive/InferScale-X-results